# Week 2, day 5 (morning) — Worksheet 10 SOLUTIONS: function arguments   (L06)

Every cell below was executed on the same Python the lab ships (3.13), and the
quoted output is what it actually printed.

Q2 and Q11 contradict the slides. Q4 and Q8 contradict what most people
assume. None of the four raises except the last.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Worksheet 10 — Function arguments. Run this once.
spring_cohort = [65, 87, 83, 53, 72, 78, 72, 78, 82, 73, 76, 74, 81]

student = {"name": "Jeff", "program": "DS", "gpa": 4.8}
grades = {"sql": 4.2, "ml": 5.5}

print(len(spring_cohort), "grades in spring_cohort")

PART A — Defaults

### Question 1

Defaults. -> `Welcome to EC2!`, then `Welcome to AWS!`, `Welcome to EC2!`, `Deploying to EC2!`.

A default makes a parameter **optional**: supply it and yours wins, leave
it out and the default fills in. Without defaults, `welcome("EC2")` raises
`TypeError: missing 1 required positional argument` — slide 24's left-hand
column.

They are filled left to right, so `welcome_d("EC2")` set `service` and left
`greeting` alone. That is why a parameter with a default cannot come before
one without: `def f(a=1, b)` is a `SyntaxError`, because there would be no
way to supply `b` positionally.

In [ ]:
def welcome(service, greeting):
    print(f"{greeting} {service}!")

welcome("EC2", "Welcome to")

def welcome_d(service="AWS", greeting="Welcome to"):
    print(f"{greeting} {service}!")

welcome_d()
welcome_d("EC2")
welcome_d("EC2", "Deploying to")

### Question 2

Slide 25's call. -> **`Welcome to Hi, welcome to!`** — the slide prints `welcome to Hi!`. Then `Hi, welcome to AWS!` from the keyword version.

A positional argument fills the **first** parameter. Always. `"Hi, welcome
to"` reads like a greeting, so the slide's author expected it to land in
`greeting`; Python put it in `service`, where it produced a grammatical
sentence about a product called "Hi, welcome to".

Nothing raised, and the output is a plausible English sentence. That is the
worst possible failure mode for an argument bug: it looks like it worked.

`welcome_d(greeting="Hi, welcome to")` is how you say what you meant, and
it is the whole argument for naming your optional arguments — Q10 again.

In [ ]:
welcome_d("Hi, welcome to")

# The slide prints `welcome to Hi!`. Python prints
# `Welcome to Hi, welcome to!`.
#
# A positional argument fills the FIRST parameter, always -- here `service`.
# There is no way for Python to guess that a string reading like a greeting
# was meant for `greeting`. If that is what you want, say so:
welcome_d(greeting="Hi, welcome to")

### Question 3

Four ways to write one call. -> `Deploying to EC2!`, four times.

All four are the **same call**. Keyword arguments are matched by name, so
their order is irrelevant; only positional arguments depend on position.

There is no fifth version with the keyword first, because
`welcome_d(greeting="x", "EC2")` is a `SyntaxError`: **positional arguments
must come before keyword arguments**. Slides 39 and 40 are two variations
on exactly that error.

Once a call has more than two or three arguments, the keyword forms are the
only ones that survive a code review — and the only ones that keep working
when someone reorders the parameters.

In [ ]:
welcome_d("EC2", "Deploying to")
welcome_d(service="EC2", greeting="Deploying to")
welcome_d(greeting="Deploying to", service="EC2")
welcome_d("EC2", greeting="Deploying to")

# All four are the same call and print the same line. Keyword arguments are
# matched by NAME, so their order does not matter -- but a positional
# argument must come before any keyword argument, which is why there is no
# fifth version with the keyword first.

PART B — The default that is not what you think

### Question 4

The mutable default. -> `['apple']`, **`['apple', 'pear']`**, **`['apple', 'pear', 'fig']`** — then the fixed version gives `['apple']` and `['pear']`.

The basket kept filling up across three separate calls that each passed no
basket at all.

**A default value is evaluated once, when the `def` line runs — not on
every call.** So `basket=[]` creates exactly one list, at definition time,
and every call that uses the default gets *that* list, complete with
whatever previous calls put in it. It survives for the lifetime of the
program.

With an immutable default — a number, a string, `None` — you never notice,
because nothing can modify it in place. With a list, a dict or a set, you
get this.

The fix is always the same: default to `None`, and build the real default
inside the body. `if basket is None: basket = []` runs on **every** call,
which is what you wanted the first time.

This is not in the deck. It is the single most common Python interview
question about functions, and it is a genuine bug you will ship if nobody
tells you.

In [ ]:
def add_item(item, basket=[]):
    basket.append(item)
    return basket

print(add_item("apple"))
print(add_item("pear"))
print(add_item("fig"))

# And the fix:
def add_item_safe(item, basket=None):
    if basket is None:
        basket = []
    basket.append(item)
    return basket

print(add_item_safe("apple"))
print(add_item_safe("pear"))

PART C — Arbitrary arguments

### Question 5

`*args`. -> `args` is a `<class 'tuple'>` of length 3, 6 and 13; the deviations are `11.718930554164631`, `12.537942414925983` and `8.750824137012644`.

`*args` **collects** every extra positional argument into a tuple. The name
is a convention, not a rule — `*nums` works identically, which is what
slide 34 was trying to demonstrate and Q11 is what happened.

A tuple, note, not a list: you cannot append to it inside the function.

`calc_standard_deviation(*spring_cohort)` does the reverse. At a **call**
site the star **spreads** a list into separate arguments, so thirteen
values arrived as thirteen arguments and `args` had length 13. Without the
star you would pass one argument that happens to be a list, `n` would be 1,
and `n - 1` would be zero — `ZeroDivisionError`.

Same symbol, opposite directions, and which one you get depends entirely on
whether you are defining or calling.

In [ ]:
def calc_standard_deviation(*args):
    print("  inside, args is a", type(args), "of length", len(args))
    n = len(args)
    mean = sum(args) / n
    squared_distance = [((num - mean) ** 2) for num in args]
    return (sum(squared_distance) / (n - 1)) ** 0.5

print(calc_standard_deviation(65, 87, 83))
print(calc_standard_deviation(65, 87, 83, 53, 72, 78))
print(calc_standard_deviation(*spring_cohort))

### Question 6

`**kwargs`. -> `kwargs is {'name': 'Jeff', 'program': 'DS', 'gpa': 4.8}` then `Name: Jeff, Program: DS, GPA: 4.8`; then `kwargs is {'name': 'Jeff', 'gpa': 4.8}` and **`Program: None`**.

`**kwargs` collects every extra **keyword** argument into a dictionary —
names and all. `*args` gets the unnamed ones, `**kwargs` gets the named
ones.

The second call is the interesting one. `program` was never passed,
`.get("program")` returned `None` rather than raising `KeyError`, and the
f-string printed the word `None` into a student record. No error, and a
report with `Program: None` in it.

That is the trade `.get()` makes, and it is the right call about half the
time. `kwargs["program"]` would have raised and told you immediately;
`.get("program", "unknown")` puts something honest in the output. Choosing
`.get()` with no default means choosing to let a gap through silently.

In [ ]:
def print_gpa(**kwargs):
    print("  kwargs is", kwargs)
    name = kwargs.get("name")
    gpa = kwargs.get("gpa")
    program = kwargs.get("program")
    print(f"Name: {name}, Program: {program}, GPA: {gpa}")

print_gpa(name="Jeff", program="DS", gpa=4.8)
print_gpa(name="Jeff", gpa=4.8)

PART D — Ordering, and what it silently changes

### Question 7

Slide 41's working order. -> `Student Name: Jeff`, `Student City: Brazil`, `Hobbies: ('Soccer', 'Poker', 'Dota')`, `Grades: {'python': 4.4, 'sql': 4.2, 'ml': 5.5}`.

The order in the definition is fixed and worth memorising:
**`name` → `*hobbies` → `location="Toronto"` → `**grades`**. Normal
parameters, then the star, then keyword-only parameters, then the
double-star.

Slides 39 and 40 show the two ways to get it wrong at the *call* site, both
`SyntaxError`: a positional argument after a keyword argument, and a
positional after `**` unpacking. Those fail at parse time — the cell will
not run at all — which is why they are not questions on this sheet.

Note `python=4.4` and `**grades` merged into one dictionary. Duplicate keys
would raise `TypeError: got multiple values for keyword argument`.

In [ ]:
def print_student_profile(name, *hobbies, location="Toronto", **grades):
    print(f"Student Name: {name}")
    print(f"Student City: {location}")
    print(f"Hobbies: {hobbies}")
    print(f"Grades: {grades}")

print_student_profile("Jeff",
                      "Soccer", "Poker", "Dota",
                      location="Brazil",
                      python=4.4, **grades)

### Question 8

The same function, one word shorter. -> `Student City: **Toronto**`, `Hobbies: ('Brazil', 'Soccer', 'Poker')`, `Grades: {}`.

A student in Toronto whose hobbies include Brazil. Nothing raised.

`*hobbies` takes **every** remaining positional argument, so there is no
position left over for `location` — a parameter after `*args` can only ever
be passed by name. It is *keyword-only*, and `"Brazil"` was just another
hobby.

That is a genuinely useful feature: it forces callers to write `location=`
and makes Q10's readability problem impossible for that parameter. It is
also completely silent when you forget, and this call is how you meet it.

Compare with worksheet 09 Q11, where too few arguments raised immediately.
Once `*args` is in the signature, that protection is gone — the function
accepts anything.

In [ ]:
print_student_profile("Jeff", "Brazil", "Soccer", "Poker")

# "Brazil" was swallowed by *hobbies, and location fell back to its default
# of "Toronto". A student in Toronto whose hobbies include Brazil.
#
# Once a parameter sits AFTER *args it becomes keyword-only -- there is no
# position left for it, because *args takes every remaining positional
# argument. That is a feature: it makes `location=` compulsory at the call
# site. It is also silent, and this call is exactly how you meet it.

### Question 9

Unpacking at the call site. -> `print_gpa(**student)` prints the same line as Q6's first call; then Ada in Lisbon with `('Chess', 'Running')` and the three merged grades.

`**student` turns each key into a keyword argument, so
`print_gpa(**{"name": "Jeff", ...})` is exactly `print_gpa(name="Jeff",
...)`. That is how you drive a function from a config dictionary or a
parsed JSON payload.

The keys have to be valid parameter names. A dictionary with `"order id"`
or `"2020"` as a key fails here — and it fails at the call, not at the
definition, which makes it easy to blame the wrong line.

Also worth seeing together: `*` spreads a list into positional arguments,
`**` spreads a dict into keyword arguments, and both can appear in the same
call in that order.

In [ ]:
print_gpa(**student)

all_grades = {"python": 4.4}
for k, v in grades.items():
    all_grades[k] = v

print_student_profile("Ada", *["Chess", "Running"],
                      location="Lisbon", **all_grades)

### Question 10

Readability. -> both calls print `rows=120 header=True overwrite=False compress=True`.

Identical to Python, and only one of them is reviewable. `export(120, True,
False, True)` in a diff tells you nothing about what got switched on, and
you cannot check it without opening the definition.

Worse, it is fragile in a way the named version is not: reorder the
parameters in the definition — a reasonable-looking tidy-up — and every
positional call silently starts meaning something else, while continuing to
run without complaint.

**Pass booleans and optional arguments by name.** Some codebases enforce it
by putting a bare `*` in the signature (`def export(rows, *, header=True,
…)`), which makes everything after it keyword-only — the same mechanism
Q8 met by accident.

In [ ]:
def export(rows, header=True, overwrite=False, compress=False):
    print(f"rows={rows} header={header} overwrite={overwrite} compress={compress}")

export(120, True, False, True)
export(120, header=True, overwrite=False, compress=True)

# Both calls are identical to Python and only the second is readable. In the
# first, `True, False, True` says nothing about what is being switched on --
# and if someone reorders the parameters next month, every positional call
# silently changes meaning while continuing to run.
#
# Rule of thumb: pass booleans and anything optional BY NAME.

### Question 11

Slide 34's rename. -> the correct version prints `11.718930554164631`, then `NameError: name 'args' is not defined`.

The parameter was renamed to `nums` and one line in the body still says
`args`. There is no `args` anywhere, so the call raises the moment that
line runs.

Slide 34 prints a result underneath this code. It cannot have come from
this code — it is the output of the version on the left of the same slide,
which still used `args` throughout. Someone renamed by eye, in one place,
and did not re-run it.

The error was kind here: `args` did not exist at all. Had there been an
`args` variable in the surrounding code — and worksheet 11 is about how
readily a function reaches out and finds one — this would not have raised.
It would have computed a standard deviation of the wrong numbers and
returned it.

In [ ]:
print(calc_standard_deviation(65, 87, 83))

def calc_std_renamed(*nums):
    n = len(nums)
    mean = sum(args) / n                                   # slide 34, as printed
    squared_distance = [((num - mean) ** 2) for num in nums]
    return (sum(squared_distance) / (n - 1)) ** 0.5

# This is SUPPOSED to raise: NameError. The parameter was renamed to `nums`
# but line 3 still says `args`, and there is no `args` anywhere.
#
# Slide 34 prints a result underneath this code. It cannot have come from
# this code -- it is the output of the version on the left of the slide,
# which still used `args` throughout. Rename with a search, not by eye.
print(calc_std_renamed(65, 87, 83))